In [0]:
%sql
USE CATALOG ecommerce;
USE SCHEMA project;

In [0]:
customers_df = spark.table("ecommerce.project.customers_clean")

products_df = spark.table("ecommerce.project.products_clean")

orders_df = spark.table("ecommerce.project.orders_clean")
customers_df.printSchema()
products_df.printSchema()
orders_df.printSchema()

In [0]:
orders_customers_df = orders_df.join(
    customers_df,
    orders_df.customer_id == customers_df.customer_id,
    "inner"
)
display(orders_customers_df)

In [0]:
sales_df = orders_customers_df.join(
    products_df,
    orders_customers_df.product_id == products_df.product_id,
    "inner"
)
display(sales_df)

In [0]:
from pyspark.sql.functions import col

In [0]:
sales_df = sales_df.withColumn(
    "sales_amount",
    col("quantity") * col("price")
)

display(sales_df)

In [0]:
sales_detail_df = sales_df.select(
    orders_df.order_id,
    orders_df.order_date,
    orders_df.customer_id,
    customers_df.customer_name,
    products_df.product_name,
    products_df.category,
    orders_df.quantity,
    products_df.price,
    col("sales_amount")
)
display(sales_detail_df)


In [0]:
sales_detail_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce.project.sales_detail")

In [0]:
from pyspark.sql.functions import sum, countDistinct

daily_sales_summary_df = sales_detail_df.groupBy(
    "order_date"
).agg(
    countDistinct("order_id").alias("total_orders"),
    sum("quantity").alias("total_quantity"),
    sum("sales_amount").alias("total_sales")
)
display(daily_sales_summary_df)
daily_sales_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce.project.daily_sales_summary")
